# Verification: Bivalent Ligand-Receptor Aggregation Without Ring Closure

Compares NFsim (no ring closure rule) against Posner et al. (1995) kinetic ODEs
(eqs 14a–14i with j₊₂ = j₋₂ = 0, i.e., no cyclic species).

**Key mappings:** k₊₁=kf, k₋₁=koff, k₊₂=kxf, k₋₂=koff.
No ring closure or opening (Z₂ = 0 for all time).

**Note:** Without ring closure, large aggregates form at high KxRT, making
NFsim slower than the ring model. Scaling factors are adjusted per dose.

In [ ]:
import subprocess, os, glob
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
import matplotlib.pyplot as plt

os.chdir(os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.')

NA = 6.02214076e23; V_cell = 1e-9; R_per_cell = 3e5; f = 0.01
V_sim = V_cell*f; RT = R_per_cell*f; ST = 2*RT
kon = 1e6; koff = 0.01; KxRT = 5
kf = kon/(NA*V_sim); kxf = KxRT/RT*koff
K1 = kf/koff; K2 = kxf/koff

LT_values = [3e5, 3e6, 1e7]
LT_labels = ['below', 'optimal', 'above']
t_end = 3000; n_steps = 300

## Posner et al. (1995) Kinetic ODEs (eqs 14a–14i, no ring terms)

With j₊₂ = j₋₂ = 0, the ring variable Z₂ = 0 always, and the jp/Z2 terms
drop out of all equations.

In [ ]:
def posner_odes_no_rings(t, y, LT_sim):
    """Posner et al. (1995) eqs 14a-14i with jp=0 (no rings).
    
    State variables:
      Y1 = total singly-bound ligand arms (one arm bound, one free)
      Y2 = total doubly-bound ligands (both arms bound, in chains)
      C01, C11, C21 = chain size distribution moments (single-complex)
      C02, C12, C22 = chain size distribution moments (two-complex)
    """
    Y1, Y2, C01, C11, C21, C02, C12, C22 = y
    S = max(ST - Y1 - 2*Y2, 0)
    L = max(LT_sim - Y1 - Y2, 0)
    
    dY1 = 2*kf*L*S - koff*Y1 - kxf*S*Y1 + 2*koff*Y2
    dY2 = kxf*S*Y1 - 2*koff*Y2
    
    dC01 = -4*kf*L*C01 + koff*C11 - 2*kxf*C01*Y1 + koff*(S - 2*C01 - C11)
    dC11 = (4*kf*L*C01 - koff*C11 - 2*kf*L*C11 + 2*koff*C21
            - kxf*C11*(Y1 + S) + koff*(S + Y1 - 2*C01 - 2*C11 - 2*C21))
    dC21 = 2*kf*L*C11 - 2*koff*C21 - 2*kxf*C21*S + koff*(Y1 - C11 - 2*C21)
    dC02 = (-4*kf*L*C02 + koff*C12 - 2*kxf*C02*Y1 + 2*kxf*C01*C11
            - 2*koff*C02 + koff*(S - 2*C01 - C11 - 2*C02 - C12))
    dC12 = (4*kf*L*C02 - koff*C12 - 2*kf*L*C12 + 2*koff*C22
            - kxf*C12*(Y1 + S) + 4*kxf*C01*C21 + kxf*C11**2 - 2*koff*C12
            + koff*(S + Y1 - 2*C01 - 2*C11 - 2*C21 - 2*C02 - 2*C12 - 2*C22))
    dC22 = (2*kf*L*C12 - 2*koff*C22 - 2*kxf*C22*S + 2*kxf*C11*C21
            - 2*koff*C22 + koff*(Y1 - C11 - 2*C21 - C12 - 2*C22))
    
    return [dY1, dY2, dC01, dC11, dC21, dC02, dC12, dC22]

ode_results = {}
for LT_val, label in zip(LT_values, LT_labels):
    LT_sim = LT_val * f
    sol = solve_ivp(posner_odes_no_rings, (0, t_end),
                    [0, 0, RT, 0, 0, 0, 0, 0],
                    args=(LT_sim,), method='LSODA', rtol=1e-8, atol=1e-10,
                    t_eval=np.linspace(0, t_end, n_steps+1))
    Y1, Y2 = sol.y[0], sol.y[1]
    ode_results[label] = dict(t=sol.t, Bonds=Y1+2*Y2,
                              S=ST-Y1-2*Y2, Y1=Y1, Y2=Y2,
                              LT_sim=LT_sim)
    print(f'{label}: Bonds_eq={Y1[-1]+2*Y2[-1]:.1f}, '
          f'S_eq={ST-Y1[-1]-2*Y2[-1]:.1f}, '
          f'Y1_eq={Y1[-1]:.1f}, Y2_eq={Y2[-1]:.1f}')

## Posner/Dembo Equilibrium (exact per-site theory)

Solve the exact per-site equilibrium relations (Posner eqs 15a-b) for
Y1 = 2·K1·L·S and Y2 = K2·S·Y1/2, together with conservation laws
S = ST - Y1 - 2·Y2 and L = LT - Y1 - Y2. This reduces to a single
nonlinear equation in S, solved by bisection. These equilibrium values
are exact (derived from detailed balance) and serve as reference lines.

In [ ]:
eq = {}
for LT_val, label in zip(LT_values, LT_labels):
    LT_sim = LT_val * f
    def f_eq(S, LT_sim=LT_sim):
        L = LT_sim / (1 + 2*K1*S + K1*K2*S**2)
        Y1 = 2*K1*L*S
        Y2 = K1*K2*L*S**2
        return S - (ST - Y1 - 2*Y2)
    S_eq = brentq(f_eq, 1e-10, ST)
    L_eq = LT_sim / (1 + 2*K1*S_eq + K1*K2*S_eq**2)
    Y1_eq = 2*K1*L_eq*S_eq
    Y2_eq = K1*K2*L_eq*S_eq**2
    Bonds_eq = Y1_eq + 2*Y2_eq
    eq[label] = dict(Bonds=Bonds_eq/ST, S=S_eq/ST)
    print(f'{label}: Bonds/ST={eq[label]["Bonds"]:.4f}, '
          f'S/ST={eq[label]["S"]:.4f}')

## Run NFsim at Three Ligand Doses

Without ring closure, large aggregates form (especially at optimal and above
doses), making NFsim slow. Scaling factor `f` is reduced for higher doses
to keep molecule counts manageable.

In [ ]:
bngl_file = 'blbr_rings_posner1995_no_rings.bngl'
with open(bngl_file) as fh:
    bngl_template = fh.read()

nf = {}
# Smaller f for higher doses to limit aggregate size and NFsim runtime
f_overrides = {'below': 0.01, 'optimal': 0.003, 'above': 0.002}
for LT_val, label in zip(LT_values, LT_labels):
    f_run = f_overrides[label]
    tmp = bngl_template.replace('LT_per_cell  3e6',
                                f'LT_per_cell  {LT_val:.0e}')
    if f_run != f:
        # Use unique suffix to avoid corrupting koff
        tmp = tmp.replace('f  0.01  # dimensionless',
                          f'f  {f_run}  # dimensionless')
    tmp = tmp.replace('suffix=>"nfr"', f'suffix=>"nfr_{label}"')
    tmp_file = f'tmp_nr_{label}.bngl'
    with open(tmp_file, 'w') as fh: fh.write(tmp)
    RT_run = R_per_cell * f_run
    LT_run = LT_val * f_run
    print(f'Running {label} (f={f_run}, RT={RT_run:.0f}, '
          f'LT={LT_run:.0f})...', end=' ', flush=True)
    subprocess.run(['bionetgen', 'run', '-i', tmp_file],
                   capture_output=True, text=True, timeout=600)
    gdat = f'tmp_nr_{label}_nfr_{label}.gdat'
    if os.path.exists(gdat):
        d = np.loadtxt(gdat, comments='#')
        nf[label] = dict(t=d[:,0], Free_L=d[:,1], Free_R=d[:,2],
                         Bonds=d[:,3], Free_L_sites=d[:,4],
                         Free_R_sites=d[:,5],
                         LT_sim=LT_run, RT_run=RT_run,
                         f_run=f_run)
        print(f'{len(d)} pts, Bonds~{d[-60:,3].mean():.0f}')
    else: print('FAILED')
for fn in glob.glob('tmp_nr_*'): os.remove(fn)

## Posner Per-Site Equilibrium Checks

At equilibrium, the per-site relations (Posner eqs 15a-b) must hold:
- Y1 = 2·K1·L·S  (capture equilibrium)
- Y2 = K2·S·Y1/2  (cross-link equilibrium)

where K1 = kf/koff, K2 = kxf/koff. These relations are exact at equilibrium
(derived from detailed balance, independent of the equivalent site approximation).

In [ ]:
print('Posner per-site equilibrium (ratio NFsim_observed / predicted):')
print(f'{"Dose":>10s} {"Y1":>8s} {"Y2":>8s}')
print('-' * 30)
for label in LT_labels:
    if label not in nf: continue
    r = nf[label]; n_eq = 60; RT_r = r['RT_run']
    L = r['Free_L'][-n_eq:].mean()
    S = r['Free_R_sites'][-n_eq:].mean()
    Y1 = r['Free_L_sites'][-n_eq:].mean() - 2*L
    Y2 = r['LT_sim'] - L - Y1
    f_r = r['f_run']; V_r = V_cell*f_r
    kf_r = kon/(NA*V_r); kxf_r = KxRT/(R_per_cell*f_r)*koff
    K1_r = kf_r/koff; K2_r = kxf_r/koff
    Y1p = 2*K1_r*L*S; Y2p = K2_r*S*Y1/2
    print(f'{label:>10s} {Y1/Y1p:8.3f} {Y2/Y2p:8.3f}')

## Comparison Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
colors = {'below': 'C0', 'optimal': 'C1', 'above': 'C2'}

for col, label in enumerate(LT_labels):
    if label not in nf:
        for row in range(2): axes[row, col].text(0.5, 0.5, 'No data',
            transform=axes[row, col].transAxes, ha='center')
        continue
    ode = ode_results[label]; r = nf[label]
    RT_r = r['RT_run']; ST_r = 2*RT_r

    # Top: Bonds (normalized to ST)
    ax = axes[0, col]
    ax.axhline(eq[label]['Bonds'], color='r', ls='--', lw=1.5,
               label='Posner equil.')
    ax.plot(ode['t'], ode['Bonds']/ST, 'k-', lw=2, label='Posner ODE')
    ax.plot(r['t'], r['Bonds']/ST_r, '-',
            color=colors[label], lw=1, alpha=0.7, label='NFsim')
    ax.set_ylabel('Bonds / ST' if col == 0 else '')
    ax.set_title(f'{label} (LT/cell={r["LT_sim"]/r["f_run"]:.0e})', fontsize=10)
    ax.legend(fontsize=8)

    # Bottom: Free receptor sites (normalized to ST)
    ax = axes[1, col]
    ax.axhline(eq[label]['S'], color='r', ls='--', lw=1.5,
               label='Posner equil.')
    ax.plot(ode['t'], ode['S']/ST, 'k-', lw=2, label='Posner ODE')
    ax.plot(r['t'], r['Free_R_sites']/ST_r, '-',
            color=colors[label], lw=1, alpha=0.7, label='NFsim')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Free R sites / ST' if col == 0 else '')
    ax.legend(fontsize=8)

fig.suptitle('Posner et al. (1995) ODEs (no rings) vs NFsim\n'
             'Dashed red = Posner equilibrium, black = Posner ODE, '
             'colored = NFsim.',
             fontsize=12)
fig.tight_layout()
fig.savefig('verify_no_rings_posner1995.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved verify_no_rings_posner1995.png')

## Error Summary

In [ ]:
print('=== Quantitative comparison (equilibrium, normalized) ===')
print(f'{"":>10s} {"Bonds/ST":>16s} {"S/ST":>16s}')
print(f'{"Dose":>10s} {"ODE":>7s} {"NFsim":>7s} '
      f'{"ODE":>7s} {"NFsim":>7s}')
print('-' * 50)
max_rel_err = 0
for label in LT_labels:
    if label not in nf: continue
    ode = ode_results[label]; r = nf[label]; n_eq = 60
    RT_r = r['RT_run']; ST_r = 2*RT_r
    B_nf = r['Bonds'][-n_eq:].mean() / ST_r
    S_nf = r['Free_R_sites'][-n_eq:].mean() / ST_r
    B_ode = ode['Bonds'][-1] / ST
    S_ode = ode['S'][-1] / ST
    err_B = abs(B_nf - B_ode) / max(B_ode, 1e-10)
    err_S = abs(S_nf - S_ode) / max(S_ode, 1e-10)
    max_rel_err = max(max_rel_err, err_B, err_S)
    print(f'{label:>10s} {B_ode:7.4f} {B_nf:7.4f} '
          f'{S_ode:7.4f} {S_nf:7.4f}')

print(f'\nMax relative error: {max_rel_err:.4f}')
if max_rel_err < 0.10:
    print('PASS: NFsim no-ring model agrees with Posner ODEs (no rings).')
    print('Note: The Posner ODEs use the equivalent site approximation (ESA)')
    print('for crosslinking kinetics. Without ring closure, larger aggregates')
    print('form and the ESA becomes less accurate, so ~5% error is expected.')
else:
    print('FAIL: Significant disagreement detected.')